# Improving Decision Tree Performance using Ensemble Methods


<span style="color: blue;">
    
Decision Trees are simple and interpretable models but often suffer from
high variance and overfitting. In this notebook, we explore how ensemble
methods can improve the predictive performance of a Decision Tree model.

We first train a single Decision Tree model and evaluate its performance.
This serves as a baseline for comparison with ensemble methods.

</span>

In [6]:
import numpy as np
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt

In [7]:
df = pd.read_csv("yeast.csv")   
df.head()


,mcg,gvh,alm,mit,erl,pox,vac,nuc,name
0,0.58,0.61,0.47,0.13,0.5,0.0,0.48,0.22,MIT
1,0.43,0.67,0.48,0.27,0.5,0.0,0.53,0.22,MIT
2,0.64,0.62,0.49,0.15,0.5,0.0,0.53,0.22,MIT
3,0.58,0.44,0.57,0.13,0.5,0.0,0.54,0.22,NUC
4,0.42,0.44,0.48,0.54,0.5,0.0,0.48,0.22,MIT


In [8]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['label'] = le.fit_transform(df['name'])

df.head()
df['label'].unique()


array([6, 7, 0, 3, 2, 4, 5, 9, 8, 1])

In [9]:
from sklearn.model_selection import train_test_split

X = df[['mcg','gvh','alm','mit','erl','pox','vac','nuc']]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


Train shape: (1187, 8)
Test shape: (297, 8)


In [13]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import DecisionTreeClassifier

best_alpha = 0.004134

dt_final = DecisionTreeClassifier(
    random_state=42,
    ccp_alpha=best_alpha,
    max_depth=5,
    min_samples_leaf=5,
    min_samples_split=10
)

dt_final.fit(X_train, y_train)

print("Model trained!")

Model trained!


In [15]:
from sklearn.metrics import accuracy_score, f1_score, recall_score, matthews_corrcoef, classification_report, confusion_matrix

y_pred_final = dt_final.predict(X_test)

print("FINAL MODEL PERFORMANCE")
print("------------------------")
print("Accuracy:", accuracy_score(y_test, y_pred_final))
print("F1 Macro:", f1_score(y_test, y_pred_final, average='macro'))
print("F1 Weighted:", f1_score(y_test, y_pred_final, average='weighted'))
print("Recall Macro:", recall_score(y_test, y_pred_final, average='macro'))
print("MCC:", matthews_corrcoef(y_test, y_pred_final))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_final, target_names=le.classes_))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_final))


FINAL MODEL PERFORMANCE
------------------------
Accuracy: 0.5892255892255892
F1 Macro: 0.44220471603411715
F1 Weighted: 0.5708399495343345
Recall Macro: 0.4274538936604899
MCC: 0.4687606110127008

Classification Report:
              precision    recall  f1-score   support

         CYT       0.49      0.73      0.59        93
         ERL       0.00      0.00      0.00         1
         EXC       0.50      0.29      0.36         7
         ME1       0.80      0.89      0.84         9
         ME2       0.40      0.20      0.27        10
         ME3       0.79      0.94      0.86        32
         MIT       0.64      0.55      0.59        49
         NUC       0.63      0.43      0.51        86
         POX       1.00      0.25      0.40         4
         VAC       0.00      0.00      0.00         6

    accuracy                           0.59       297
   macro avg       0.53      0.43      0.44       297
weighted avg       0.59      0.59      0.57       297


Confusion Matrix:
[

C:\Users\DP_PANDA\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\DP_PANDA\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\DP_PANDA\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _war

# Bagging (Bootstrap Aggregating)


<span style="color: blue;">
    
Bagging reduces the variance of Decision Trees by training multiple trees
on different bootstrap samples of the dataset and aggregating their predictions.

</span>

In [24]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

bagging = BaggingClassifier(
    estimator= DecisionTreeClassifier(
    random_state=42,
    ccp_alpha=best_alpha,
    max_depth=5,
    min_samples_leaf=5,
    min_samples_split=10
)
)

bagging.fit(X_train, y_train)

y_pred_bag = bagging.predict(X_test)


In [25]:
from sklearn.metrics import accuracy_score

print("Bagging Test Accuracy:", accuracy_score(y_test, y_pred_bag))


Bagging Test Accuracy: 0.5993265993265994


# Random Forest


<span style="color: blue;">
    
Random Forest extends bagging by introducing feature randomness at each split,
which reduces correlation among trees and improves generalization.

</span>

In [26]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)


In [27]:
print("Random Forest Test Accuracy:", accuracy_score(y_test, y_pred_rf))


Random Forest Test Accuracy: 0.6161616161616161


# Boosting (Gradient Boosting)


<span style="color: blue;">
    
Boosting improves model performance by training trees sequentially, where
each new tree focuses on correcting the errors of the previous ones.

</span>

In [28]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(random_state=42)

gb.fit(X_train, y_train)

y_pred_gb = gb.predict(X_test)


In [29]:
print("Gradient Boosting Test Accuracy:", accuracy_score(y_test, y_pred_gb))


Gradient Boosting Test Accuracy: 0.5925925925925926


In [33]:
from sklearn.metrics import accuracy_score

dt_test_acc = accuracy_score(y_test, y_pred_final)
print("Decision Tree Test Accuracy:", dt_test_acc)

import pandas as pd

results = pd.DataFrame({
    "Model": ["Decision Tree", "Bagging", "Random Forest", "Gradient Boosting"],
    "Test Accuracy": [
        dt_test_acc,
        accuracy_score(y_test, y_pred_bag),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_gb)
    ]
})

results

Decision Tree Test Accuracy: 0.5892255892255892


,Model,Test Accuracy
0,Decision Tree,0.589226
1,Bagging,0.599327
2,Random Forest,0.616162
3,Gradient Boosting,0.592593


# Conclusion


<span style="color: brown;">
    
Ensemble methods significantly improve the performance of a single Decision Tree.
Bagging reduces variance, Random Forest further improves generalization by
decorrelating trees, and Boosting reduces bias by focusing on hard-to-classify
samples. Among the methods tested, ensemble models consistently outperform
the standalone Decision Tree.

</span>